# Conversor OCR a Crédito — Google Colab

Este notebook incluye el código y la plantilla: **sólo necesitas subir los Excel del OCR**.
Lee los archivos directamente de la carpeta actual (`Path.cwd()`, normalmente `/content`).
No necesitas escribir direcciones ni montar Google Drive.

Ejecuta las celdas de arriba abajo. Si ya subiste los Excel en el panel **Archivos**, omite la celda 2.
Se genera un Excel por archivo, junto con trazabilidad, conciliación, revisión y un resumen del lote.
Los resultados se guardan en una subcarpeta nueva en cada ejecución para evitar sobrescrituras.

**Duración:** los cortes intermedios de los ejemplos informan 12 meses. El modo predeterminado no
anualiza las razones de esos periodos hasta confirmar su duración. La conciliación contable sí se realiza.
Los datos temporales de Colab pueden perderse al cerrar la sesión: descarga el ZIP al terminar.


## 1. Instalar la dependencia

In [ ]:
%pip -q install "openpyxl>=3.1,<4"


## 2. Subir los Excel (opcional si ya están en Archivos)
Selecciona todos los archivos OCR que quieres convertir. No hace falta subir la plantilla ni el script anterior.

In [ ]:
from google.colab import files
archivos_subidos = files.upload()


## 3. Cargar el conversor y la plantilla
Ejecuta esta celda sin modificarla. Incluye la versión probada con los 20 archivos y la adaptación para Colab.

In [ ]:
#!/usr/bin/env python3
"""Convierte exportaciones Accounts (MMAS) al formato Output.xlsx de Crédito.

Python >=3.10; pip install openpyxl
No ejecuta OCR: transforma el Excel ya clasificado por la herramienta.
Ver README.md antes de aprobar las reclasificaciones propuestas.
"""
from __future__ import annotations
import argparse
import csv
import json
import math
import re
from datetime import datetime, date
from pathlib import Path
from decimal import Decimal, InvalidOperation
import openpyxl
from openpyxl.cell.cell import MergedCell
from openpyxl.comments import Comment
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.workbook.properties import CalcProperties
from openpyxl.utils import get_column_letter

# Código: coeficiente. Sólo cuentas de detalle, nunca subtotal + sus hijos.
BG = {
 9: {'1050':1,'1100':1,'1150':1}, 10:{'1200':1},
 11:{'1250':1}, 12:{'1400':1,'1450':1,'1500':1,'1550':1,'1600':1},
 13:{'1230':1,'1240':1}, 15:{'1700':1}, 17:{'1790':1},
 21:{'1670':1,'1680':1}, 24:{'1650':1,'1350':1,'1800':1},
 29:{'2050':1,'2100':1},30:{'2150':1},31:{'2300':1},32:{'2200':1},
 33:{'2070':1,'2250':1,'2350':1,'2360':1},34:{'2400':-1},
 40:{'2650':1,'2850':1,'2900':1,'2820':-1},
 42:{'2550':1},43:{'2700':1},
 44:{'2450':1,'2500':1,'2530':1,'2560':1,'2580':1,'2600':1,'2800':1,'2740':1,'2750':1},
 49:{'3400':1},50:{'3070':1,'3100':1},51:{'3500':1},
 52:{'3450':1,'3460':1},53:{'9951':1,'3900':1,'3920':1},
 54:{'3200':1,'3250':1,'3300':1,'3350':1},
 56:{'3550':1,'3750':1},57:{'3700':1},58:{'3850':1},59:{'3150':1},
 60:{'3050':1,'3360':1,'3370':1,'3600':1,'3650':1,'3800':1,'3910':1,'3960':1,'3965':1,'3970':1,'3940':1,'3950':1},
 65:{'4050':1,'4100':1,'4150':1,'4200':1,'4210':1},
 66:{'9952':1,'4400':1,'4250':1,'4300':1,'4340':1,'4345':1,'4350':1,'4600':1,'4660':1,'4680':1,'4700':1,'4800':1},
 68:{'4390':1},69:{'4450':1},70:{'4220':1,'4230':1,'4330':1},
 76:{'5050':1,'5100':1,'5150':1},77:{'5160':1},
 80:{'4650':1,'4670':1,'4690':1,'4750':1,'4850':1,'5180':1,'5190':1,'5420':1,'5450':-1},
 81:{'5400':1,'5410':1},84:{'5170':1},
}
ER = {
 7:{'6050':1,'6150':-1,'6200':-1},8:{'6250':1,'6350':1},
 12:{'6420':1,'6450':1,'6470':1,'6490':1,'6500':1,'6550':1,'6560':1,'6565':1,'6570':1,'6600':1,'6620':1,'6650':1,'6580':1,'6700':1,'6750':1,'7530':1,'6400':-1},
 16:{'6850':1,'6870':1,'6880':1},17:{'6800':1},
 23:{'6900':1,'7000':1,'7010':-1,'7040':1,'9953':1,'7050':1,'7090':-1,'9954':-1,'7100':-1,'7120':-1,'7150':1},
 27:{'7250':1,'7320':-1},28:{'7300':1},
 37:{'7230':-1},52:{'6350':1,'6600':1,'6650':1},
}
BG_INPUTS = list(range(9,25))+list(range(29,36))+list(range(40,46))+list(range(49,62))+list(range(65,71))+list(range(76,85))
ER_INPUTS = [7,8,12,16,17,18,19,23,27,28,29,33,37,38,41,42,43,52]
PERIOD_COLS = [2,4,6,8,10,18,20]

def number(value):
    """Números en convención MX/US: coma de miles y punto decimal.
    Vacío NO equivale a cero. Valores ambiguos o inválidos detienen el proceso.
    """
    if value is None or value == '':
        raise ValueError('Importe vacío; confirmar en origen, no sustituir por cero.')
    if isinstance(value, bool):
        raise ValueError('Un booleano no es un importe.')
    if isinstance(value, (float,int)):
        if not math.isfinite(value): raise ValueError('Importe no finito')
        return Decimal(str(value))
    t=str(value).strip().replace('\u00a0','').replace(' ','')
    t=re.sub(r'^(MXN|USD|\$)', '', t, flags=re.I)
    if t in ('-','—','–'): return Decimal(0)
    if t.startswith('(') and t.endswith(')'): t='-'+t[1:-1]
    if not re.fullmatch(r'[+-]?(?:\d+|\d{1,3}(?:,\d{3})+)(?:\.\d+)?',t):
        raise ValueError(f'Importe no reconocido: {value!r}')
    return Decimal(t.replace(',',''))

def as_date(value):
    if isinstance(value, datetime): return value.date()
    if isinstance(value, date): return value
    for fmt in ('%m/%d/%Y','%Y-%m-%d','%d/%m/%Y'):
        try: return datetime.strptime(str(value).strip(),fmt).date()
        except ValueError: pass
    raise ValueError(f'Fecha no reconocida: {value!r}')

def read_source(path):
    w=openpyxl.load_workbook(path,data_only=True)
    if 'Accounts' not in w: raise ValueError('Se requiere la hoja Accounts.')
    s=w['Accounts']
    if s['B6'].value != 'Statement Date': raise ValueError('Encabezado OCR distinto del formato soportado.')
    periods={}
    for col in range(3,s.max_column+1):
        if s.cell(6,col).value is None: continue
        dt=as_date(s.cell(6,col).value)
        if dt in periods: raise ValueError(f'Fecha duplicada: {dt}')
        if s.cell(39,col).value != 'MXN': raise ValueError('Sólo se soporta MXN; no se aplica conversión cambiaria.')
        units=str(s.cell(38,col).value).strip().lower()
        factors={'actual':1,'thousands':1000,'millions':1000000}
        if units not in factors: raise ValueError(f'Unidad no soportada: {units}')
        entries={}; section=None
        for row in range(45,s.max_row+1):
            label=str(s.cell(row,2).value or '').strip()
            if label in ('Balance Sheet','Income Statement','Cash Flow'): section=label
            code=str(s.cell(row,1).value or '').strip()
            if section not in ('Balance Sheet','Income Statement') or not code: continue
            if code.endswith('.0'): code=code[:-2]
            key=(section,code)
            if key in entries: raise ValueError(f'Cuenta duplicada en {section}: {code}')
            try: val=number(s.cell(row,col).value)*factors[units]/Decimal(1000)
            except ValueError as ex: raise ValueError(f'{s.title}!{get_column_letter(col)}{row}: {ex}') from ex
            entries[key]=(val,label,f'Accounts!{get_column_letter(col)}{row}')
        months=number(s.cell(7,col).value)
        if months != int(months) or not 1<=months<=12:
            raise ValueError(f'Número de meses inválido: {months}')
        periods[dt]={'entries':entries,'months':int(months)}
    if not periods: raise ValueError('No se encontraron periodos.')
    return str(s['B1'].value), periods

def convert(entrada, plantilla, salida, config=None, acumulado_enero=False):
    entrada,plantilla,salida=map(Path,(entrada,plantilla,salida))
    if salida.resolve() in (entrada.resolve(),plantilla.resolve()): raise ValueError('La salida debe ser un archivo nuevo.')
    if salida.exists(): raise FileExistsError(f'La salida ya existe: {salida}')
    company,periods=read_source(entrada)
    w=openpyxl.load_workbook(plantilla,keep_links=False)
    for name in ('Bal. Gral.','Edo. Res.'):
        if name not in w: raise ValueError(f'Falta hoja {name}')
    bg,er=w['Bal. Gral.'],w['Edo. Res.']
    if bg['A46'].value!='ACTIVO TOTAL' or er['A7'].value!='Ventas Netas':
        raise ValueError('La versión de la plantilla no coincide con el esquema soportado.')
    mapping={'Bal. Gral.':BG,'Edo. Res.':ER}
    if config:
        for sh, rules in json.loads(Path(config).read_text(encoding='utf-8')).items():
            if sh not in mapping: raise ValueError(f'Hoja no permitida: {sh}')
            mapping[sh]=dict(mapping[sh])
            for r,terms in rules.items():
                if int(r) not in (BG_INPUTS if sh=='Bal. Gral.' else ER_INPUTS): raise ValueError(f'Fila no editable: {sh}!{r}')
                if sh=='Bal. Gral.' and int(r) in (81,83): raise ValueError('Las filas de capital 81/83 usan una regla especial; modifique convert, no el JSON.')
                mapping[sh][int(r)]={str(k):Decimal(str(v)) for k,v in terms.items()}
    for sh,rules in mapping.items():
        seen={}
        for r,terms in rules.items():
            if sh=='Edo. Res.' and r==52: continue  # Cédula informativa D&A.
            for code,coef in terms.items():
                if not Decimal(str(coef)).is_finite(): raise ValueError('Coeficiente no finito.')
                if code in seen: raise ValueError(f'Cuenta {code} duplicada en {sh}: filas {seen[code]} y {r}.')
                seen[code]=r
    ordered=sorted(periods)
    if len(ordered)>5: raise ValueError('Máximo cinco periodos por archivo.')
    slots=[2,4,6,8,10] if ordered[-1].month!=12 or len(ordered)==5 else [2,4,6,8]
    dates=dict(zip(ordered,slots[-len(ordered):]))
    active={dates[d] for d in periods}
    # Limpiar datos de otras empresas/periodos; conservar estilos y estructura.
    for s in (bg,er):
        for row in s:
            for cell in row:
                if isinstance(cell,MergedCell): continue
                cell.comment=None
                if isinstance(cell.value,str) and cell.value.startswith('='):
                    cell.value=cell.value.replace('[1]','')
        for c in PERIOD_COLS:
            if c not in active:
                for row in s.iter_rows(min_row=4,max_col=c+1,min_col=c):
                    for cell in row:
                        if not isinstance(cell,MergedCell): cell.value=None
        s['A1']=company
    bg['O2']=None; bg['O1']=None
    for a in ('M91','O91','M95','O95'): bg[a]=None
    # Eliminar comparaciones cuyo periodo no se recibió.
    for sh, specs in [(bg,[(13,4,6),(14,6,8),(15,8,10),(23,18,20)]),
                      (er,[(14,4,6),(15,6,8),(16,8,10),(24,18,20)])]:
        for c,left,right in specs:
            if left not in active or right not in active:
                for row in sh.iter_rows(min_row=4,max_row=90 if sh==bg else 57,min_col=c,max_col=c):
                    if not isinstance(row[0],MergedCell): row[0].value=None
    audit=[]; checks=[]; warnings=[]
    for dt,p in sorted(periods.items()):
        col=dates[dt]; letter=get_column_letter(col); entries=p['entries']
        months=p['months']
        if acumulado_enero:
            months=dt.month
            warnings.append([str(dt),'Supuesto solicitado',f'Resultados acumulados desde enero: se usan {months} meses; OCR informa {p["months"]}.'])
        elif dt.month!=12 and months==12:
            months=None
            warnings.append([str(dt),'Duración por confirmar','El OCR indica 12 meses para un corte distinto de diciembre. Se omiten razones que necesitan anualización. Use --acumulado-enero sólo si confirma resultados desde enero a cada corte.'])
        p['months_used']=months
        def get(section,code):
            if (section,code) not in entries: raise ValueError(f'Falta cuenta requerida {section}/{code} para {dt}')
            return entries[(section,code)][0]
        values={}
        for sheet,section,inputs in [(bg,'Balance Sheet',BG_INPUTS),(er,'Income Statement',ER_INPUTS)]:
            for r in inputs:
                sheet.cell(r,col,0)
            vals={r:Decimal(0) for r in inputs}
            for r,terms in mapping[sheet.title].items():
                vals[r]=sum((get(section,k)*Decimal(str(coef)) for k,coef in terms.items()),Decimal(0))
                sources=[]
                for code,coef in terms.items():
                    v,label,origin=entries[(section,code)]
                    audit.append([str(dt),sheet.title,f'{letter}{r}',code,label,origin,float(v),float(coef),float(v*Decimal(str(coef)))])
                    sources.append(f'{code}: {origin} × {coef}')
                sheet.cell(r,col).comment=Comment('\n'.join(sources),'Conversor OCR')
            if sheet==bg:
                net=get('Income Statement','3300155')
                vals[83]=net
                # El capital fuente incluye resultado del año en 5400/5410.
                vals[81]-=net
                for rr,sign in [(81,-1),(83,1)]:
                    audit.append([str(dt),sheet.title,f'{letter}{rr}','3300155','Separar resultado del año del acumulado',entries[('Income Statement','3300155')][2],float(net),sign,float(net*sign)])
                for r in [81,83]: sheet.cell(r,col).comment=Comment('Capital: 5400 + 5410 menos utilidad neta = ejercicios anteriores; utilidad neta se presenta por separado. Confirmar clasificación contable.','Conversor OCR')
            for r,v in vals.items():
                sheet.cell(r,col,float(v)).number_format='#,##0.000;[Red](#,##0.000);–'
            values[sheet.title]=vals
        b,e=values['Bal. Gral.'],values['Edo. Res.']
        bg.cell(4,col,datetime.combine(dt,datetime.min.time()))
        bg.cell(5,col,'Por validar'); bg.cell(6,col,f'OCR; meses: {months if months is not None else "por confirmar"}; dictaminador no informado')
        bg.cell(95,col).value=None
        if col-2 not in active:
            for r in (93,94): bg.cell(r,col).value=None
        er.cell(4,col,f"='Bal. Gral.'!{letter}4")
        er.cell(5,col,f"='Bal. Gral.'!{letter}5")
        write_ratios(bg,letter,col,months)
        er.cell(38,col,f'={letter}35+{letter}37')
        for r in (41,42,43): er.cell(r,col).value=None
        current=sum(b[r] for r in range(9,25)); fixed=sum(b[r] for r in range(29,36))
        assets=current+fixed+sum(b[r] for r in range(40,46))
        short=sum(b[r] for r in range(49,62)); long=sum(b[r] for r in range(65,71))
        equity=sum(b[r] for r in range(76,85))
        gross=e[7]-e[8]; op=gross-e[12]; pre=op-(e[16]-e[17]+e[18]+e[19])+e[23]
        profit=pre-e[27]-e[28]-e[29]+e[33]
        for label,actual,section,code in [
            ('Activo circulante',current,'Balance Sheet','3300244'),
            ('Activo fijo neto',fixed,'Balance Sheet','3300544'),
            ('Activo total',assets,'Balance Sheet','3300144'),
            ('Pasivo circulante',short,'Balance Sheet','3301144'),
            ('Pasivo LP',long,'Balance Sheet','3301344'),
            ('Capital',equity,'Balance Sheet','3301444'),
            ('Utilidad bruta',gross,'Income Statement','3300555'),
            ('Utilidad operativa',op,'Income Statement','3300455'),
            ('Resultado antes impuestos',pre,'Income Statement','3300355'),
            ('Resultado neto',profit,'Income Statement','3300155')]:
            source=get(section,code); diff=actual-source
            checks.append([str(dt),label,float(source),float(actual),float(diff),'OK' if abs(diff)<=Decimal('0.001') else 'REVISAR'])
        diff=assets-short-long-equity
        checks.append([str(dt),'Activo - pasivo - capital',0,float(diff),float(diff),'OK' if abs(diff)<=Decimal('0.001') else 'REVISAR'])
        used={(sec,code) for sh,sec in [('Bal. Gral.','Balance Sheet'),('Edo. Res.','Income Statement')] for terms in mapping[sh].values() for code in terms}
        for (section,code),(v,label,origin) in entries.items():
            if v and (section,code) not in used and not code.startswith(('330','220')):
                warnings.append([str(dt),'Cuenta no mapeada',f'{origin}: {code} {label} = {v} miles; verificar cobertura.'])
    notes=[
        'El OCR no permite replicar todas las reclasificaciones manuales del ejemplo. La conciliación no valida la clasificación.',
        'Sin desglose: 1250 se presenta en cuentas por cobrar, 1700 en pagos anticipados, 3500 en acreedores diversos; impuestos a favor no se separan en IVA/ISR.',
        '7000/7010 no identifican explícitamente resultado cambiario: se presentan como otros ingresos/gastos; no se presume su naturaleza.',
        '5100 se conserva en capital; no se infieren aportaciones para futuros aumentos ni se trasladan al pasivo.',
        '5400/5410 se agrupan y se separa la utilidad del año; confirmar esta convención con los estados financieros originales.',
        '3700 se asigna provisionalmente a participación de trabajadores; 3750 a sueldos. Confirmar naturaleza de ambas cuentas.',
        'La depreciación de resultados usa 6350+6600+6650; no usa cambio en depreciación acumulada. No separa deterioro incluido en 6350.',
        'Ceros sin cuenta asignada son categorías no desglosadas por el OCR, no evidencia de inexistencia. Buró, datos fiscales y dictamen no se infieren.',
        'Sólo se presentan periodos recibidos en el panel principal; el panel lateral R/T queda vacío. Excel recalcula fórmulas al abrir; openpyxl no calcula ni guarda resultados de fórmulas.',
        'Los controles y la trazabilidad documentan la ejecución del conversor; si modifica cifras manualmente, vuelva a conciliar.',
    ]
    warnings.extend(['Todos','Revisión de criterio',n] for n in notes)
    for name,headers,rows in [
        ('Conciliacion',['Fecha','Control','OCR (miles MXN)','Convertido (miles MXN)','Diferencia','Estado'],checks),
        ('Trazabilidad',['Fecha','Hoja destino','Celda destino','Código','Descripción OCR','Celda origen','Miles MXN','Coeficiente','Contribución'],audit),
        ('Revision',['Fecha','Tipo','Observación'],warnings)]:
        if name in w: del w[name]
        s=w.create_sheet(name); s.append(headers)
        for row in rows: s.append(row)
        s.freeze_panes='A2';s.auto_filter.ref=s.dimensions
        for cell in s[1]: cell.font=Font(color='FFFFFF',bold=True);cell.fill=PatternFill('solid',fgColor='17365D')
        for c in range(1,len(headers)+1): s.column_dimensions[get_column_letter(c)].width=22 if c!=3 or name!='Revision' else 110
        for row in s.iter_rows(min_row=2):
            for cell in row:
                if isinstance(cell.value,float): cell.number_format='#,##0.000;[Red](#,##0.000)'
                cell.alignment=Alignment(vertical='top',wrap_text=True)
        if name=='Revision':
            for rr in range(2,s.max_row+1): s.row_dimensions[rr].height=60
    w.calculation=CalcProperties(calcId=0,fullCalcOnLoad=True,forceFullCalc=True,calcMode='auto')
    # No quedan enlaces externos ni referencia a la carátula ausente.
    er['K2']="='Bal. Gral.'!A3"
    for s in w:
        for row in s:
            for cell in row:
                if cell.data_type=='f' and ('[' in cell.value or 'Carátula!' in cell.value):
                    raise ValueError(f'Referencia no resuelta: {s.title}!{cell.coordinate}')
    salida.parent.mkdir(parents=True,exist_ok=True)
    w.save(salida)
    report={'version':'2.0','empresa':company,'fuente':entrada.name,'unidad':'miles MXN',
            'periodos':[{'fecha':str(dt),'columna':get_column_letter(dates[dt]),'meses_ocr':p['months'],'meses_usados':p['months_used']} for dt,p in sorted(periods.items())],
            'controles':checks,'advertencias':warnings}
    salida.with_suffix('.revision.json').write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')
    return report

def write_ratios(bg,letter,col,months):
    """Razones homogéneas, independientemente de la columna del corte."""
    ref=lambda r: f"'Edo. Res.'!{letter}{r}"
    bg.cell(98,col,f'=IF({letter}62=0,"NA",{letter}26/{letter}62)')
    bg.cell(99,col,f'=IF({letter}62=0,"NA",({letter}26-{letter}12)/{letter}62)')
    bg.cell(100,col,f'=IF({letter}85=0,"NA",{letter}73/{letter}85)')
    bg.cell(101,col,f'=IF({ref(16)}<=0,"NA",{ref(14)}/{ref(16)})')
    bg.cell(104,col,f'=IF({ref(7)}=0,"NA",{ref(35)}/{ref(7)})')
    annual_rows=(102,105,106,108,109,110,111)
    if months is None:
        for r in annual_rows:
            bg.cell(r,col,'POR CONFIRMAR MESES')
            bg.cell(r,col).comment=Comment('OCR: 12 meses para corte intermedio. Confirmar duración antes de anualizar.','Conversor OCR')
        return
    factor=f'12/{months}'
    ebitda=f'({ref(14)}+{ref(52)})*{factor}'
    sales=f'({ref(7)}*{factor})'; cost=f'({ref(8)}*{factor})'
    bg.cell(102,col,f'=IF({letter}92<=0,0,IF(({ebitda})<=0,"NA",{letter}92/({ebitda})))')
    bg.cell(105,col,f'=IF({letter}46=0,"NA",{ref(35)}*{factor}/{letter}46)')
    bg.cell(106,col,f'=IF({letter}85=0,"NA",{ref(35)}*{factor}/{letter}85)')
    bg.cell(108,col,f'=IF({sales}<=0,"NA",360*({letter}10+{letter}11)/{sales})')
    bg.cell(109,col,f'=IF({cost}<=0,"NA",360*{letter}49/{cost})')
    bg.cell(110,col,f'=IF({cost}<=0,"NA",360*{letter}12/{cost})')
    bg.cell(111,col,f'=IF(COUNT({letter}108:{letter}110)=3,{letter}108-{letter}109+{letter}110,"NA")')


import base64
import tempfile
import zipfile
from datetime import datetime

PLANTILLA_BASE64 = """..."""

def ejecutar_colab(acumulado_enero=False):
    """Busca archivos .xlsx en Path.cwd(); no pide rutas ni usa argumentos CLI."""
    carpeta = Path.cwd()
    candidatos = sorted(p for p in carpeta.glob("*.xlsx") if not p.name.startswith("~$"))
    if not candidatos:
        raise ValueError("No hay Excel en la carpeta actual. Súbelos con la celda de carga o el panel Archivos.")
    salida = carpeta / ("resultados_credito_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
    salida.mkdir()
    resumen=[]
    procesados=0
    with tempfile.TemporaryDirectory(prefix="plantilla_credito_") as temporal:
        plantilla=Path(temporal)/"Plantilla_Credito.xlsx"
        plantilla.write_bytes(base64.b64decode(PLANTILLA_BASE64))
        for archivo in candidatos:
            try:
                libro=openpyxl.load_workbook(archivo,read_only=True,data_only=True)
                es_ocr="Accounts" in libro.sheetnames
                libro.close()
                if not es_ocr:
                    resumen.append([archivo.name,"OMITIDO: no contiene Accounts",0,0,0,0,"",""])
                    print(f"Omitido: {archivo.name} (no es exportación Accounts)")
                    continue
                destino=salida/(archivo.stem+"_credito.xlsx")
                reporte=convert(archivo,plantilla,destino,acumulado_enero=acumulado_enero)
                diferencias=sum(c[-1]!="OK" for c in reporte["controles"])
                pendientes=sum(p["meses_usados"] is None for p in reporte["periodos"])
                estado="REVISAR DIFERENCIAS" if diferencias else "CONCILIA; REVISAR CRITERIOS"
                resumen.append([archivo.name,estado,len(reporte["periodos"]),len(reporte["controles"]),diferencias,pendientes,destino.name,""])
                procesados+=1
                print(f"{archivo.name}: {estado}; {pendientes} periodos con meses por confirmar.")
            except Exception as error:
                resumen.append([archivo.name,"ERROR",0,0,0,0,"",str(error)])
                print(f"ERROR en {archivo.name}: {error}")
    with (salida/"resumen_lote.csv").open("w",encoding="utf-8-sig",newline="") as stream:
        writer=csv.writer(stream)
        writer.writerow(["Archivo","Estado","Periodos","Controles","Diferencias","Meses pendientes","Salida","Error"])
        writer.writerows(resumen)
    paquete=salida.with_suffix(".zip")
    with zipfile.ZipFile(paquete,"w",zipfile.ZIP_DEFLATED) as z:
        for archivo in sorted(salida.iterdir()):
            z.write(archivo,archivo.name)
    errores=sum(f[1]=="ERROR" for f in resumen)
    print(f"\nTerminó: {procesados} convertidos, {errores} errores. Revisa resumen_lote.csv.")
    if procesados==0:
        print("No se generaron conversiones. El ZIP contiene el diagnóstico.")
    print(f"Carpeta de entrada: {carpeta}")
    print(f"ZIP generado: {paquete.name}")
    return paquete, resumen


## 4. Convertir los archivos de la carpeta actual

Deja `ACUMULADO_DESDE_ENERO = False` si aún no confirmas la duración.
Cámbialo a `True` **sólo si TODOS los archivos contienen resultados acumulados desde enero**:
marzo usará 3 meses, junio 6, agosto 8, etc. No cambia los importes contables.

El proceso revisa sólo `.xlsx` de la carpeta actual, sin subcarpetas. Los archivos sin hoja
`Accounts` se omiten y los errores individuales se registran sin interrumpir los demás.
Si existen Excel OCR de otra tarea en la misma carpeta, también se procesarán; retíralos antes de ejecutar.


In [ ]:
ACUMULADO_DESDE_ENERO = False

archivo_zip, resumen = ejecutar_colab(acumulado_enero=ACUMULADO_DESDE_ENERO)


## 5. Descargar los resultados
Abre los Excel para recalcular las fórmulas. Revisa **Conciliacion** y **Revision** antes de usarlos. Los controles de conciliación documentan la ejecución; si modificas cifras, vuelve a conciliar.

La descarga incluye `resumen_lote.csv`, los Excel y sus reportes JSON. Si el navegador bloquea la descarga, usa el panel Archivos para descargar el ZIP generado.

In [ ]:
from google.colab import files
files.download(str(archivo_zip))


## Alcance

- Conversor de exportaciones MMAS con hoja Accounts, no OCR de imágenes/PDF.
- Sólo MXN; convierte unidades a miles de pesos y admite hasta cinco periodos por archivo.
- Conserva las reglas y alertas de clasificación de la versión 2.0. No replica ajustes manuales que no estén en el OCR.
- 5400/5410 y los ajustes 7650 requieren revisión de su naturaleza; no agrega ajustes para forzar el balance.
- Los periodos cuya duración es dudosa muestran `POR CONFIRMAR MESES` en las razones que anualizan.
- openpyxl conserva las fórmulas pero no las calcula; Excel las recalcula al abrir. La conciliación se calcula en Python.
- Una conciliación correcta no sustituye la validación del OCR contra los documentos originales.
